# Train and compare all models

Runs the twelve (model, feature set) combinations with the rolling-origin protocol from `energyforecast.training` and plots the forecasts. A full run over 2020-2021 takes hours on CPU; set `MAX_PERIODS` to something small to try it out.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from energyforecast.data import load_dataset
from energyforecast.training import GRID, make_forecaster, run_grid

%matplotlib inline

## Data

The first four years (35,064 hours, 2016-2019) form the initial training window; 2020-2021 is the test period.

In [ ]:
data = load_dataset("../data")
N_TRAIN = 35064
MAX_PERIODS = 10  # None for the full test period

data[["total_aggregated", "business_hour", "saturday", "sunday"]].head()

## One model

In [ ]:
forecaster = make_forecaster("cnn", "business_hour", seed=0)
result = forecaster.run(data, n_train=N_TRAIN, max_periods=MAX_PERIODS)
result.summary()

In [ ]:
def plot_result(result, n_hours=None):
    frame = result.frame()
    if n_hours is not None:
        frame = frame.iloc[:n_hours]
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(frame["truth"], label="observed")
    ax.plot(frame["prediction"], label="forecast")
    if "lower" in frame:
        ax.fill_between(frame.index, frame["lower"], frame["upper"], alpha=0.3, label="interval")
    ax.set_xlabel("hours into the test period")
    ax.set_ylabel("MW")
    ax.set_title(f"{result.model} / {result.feature_set}: rmse {result.rmse:.0f}")
    ax.legend()
    return ax

plot_result(result);

## Bayesian MLP

The interval is mean ± 3 standard deviations over 100 weight samples. It only reflects uncertainty in the weights, not observation noise, so coverage is well below 99%.

In [ ]:
bayes = make_forecaster("bmlp", "business_hour", window=24, seed=0)
bayes_result = bayes.run(data, n_train=N_TRAIN, max_periods=MAX_PERIODS)
plot_result(bayes_result);

## Full grid

In [ ]:
table, results = run_grid(data, GRID, n_train=N_TRAIN, max_periods=MAX_PERIODS, verbose=False, seed=0)
table

In [ ]:
ax = table.plot.barh(x="model", y="rmse", legend=False, figsize=(8, 5))
ax.set_yticklabels(table["model"] + " / " + table["features"])
ax.set_xlabel("RMSE (MW)")
ax.invert_yaxis();

In [ ]:
fig, axes = plt.subplots(len(results), 1, figsize=(14, 3 * len(results)), sharex=True)
for ax, ((model, features), res) in zip(axes, results.items()):
    frame = res.frame().iloc[:24 * 7]
    ax.plot(frame["truth"], label="observed")
    ax.plot(frame["prediction"], label="forecast")
    ax.set_title(f"{model} / {features}: rmse {res.rmse:.0f}", loc="left")
axes[0].legend()
fig.tight_layout()

In [ ]:
table.to_csv("../results/summary.csv", index=False)